# Download Phi-3 Small/Medium vao Google Drive

Notebook nay chi dung de tai model ve Google Drive, khong load model len RAM/VRAM.

Thu muc dich mac dinh:

```text
/content/drive/MyDrive/hf_cache
```

Sau khi tai xong, Drive se co cau truc kieu Hugging Face cache nhu `hf_cache/hub`, `hf_cache/modules`, `hf_cache/xet`.

In [ ]:
# Cai thu vien tai model. Notebook nay khong can transformers vi chi download weights.
%pip -q install -U huggingface_hub hf_xet

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import os
import shutil

DRIVE_ROOT = Path("/content/drive/MyDrive")
HF_CACHE_DIR = DRIVE_ROOT / "hf_cache"
HF_HUB_CACHE = HF_CACHE_DIR / "hub"

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)

# De Hugging Face tao dung layout: hf_cache/hub, hf_cache/modules, hf_cache/xet.
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "transformers")

total, used, free = shutil.disk_usage(DRIVE_ROOT)
gb = 1024 ** 3

print("Drive root:", DRIVE_ROOT)
print("HF cache:", HF_CACHE_DIR)
print("HF hub cache:", HF_HUB_CACHE)
print(f"Drive free: {free / gb:.1f} GB / {total / gb:.1f} GB")

## Model se tai

Bo nay bo qua Mini vi ban da co san, chi tai 2 ban short-context con lai cua Phi-3 text-only:

- `microsoft/Phi-3-small-8k-instruct`
- `microsoft/Phi-3-medium-4k-instruct`

Neu muon tai ban 128K, doi danh sach `MODEL_IDS` o cell ben duoi.

In [ ]:
MODEL_IDS = [
    "microsoft/Phi-3-small-8k-instruct",
    "microsoft/Phi-3-medium-4k-instruct",
]

for model_id in MODEL_IDS:
    print(model_id)

In [ ]:
from datetime import datetime, timezone
import json

from huggingface_hub import snapshot_download

downloaded = []

for repo_id in MODEL_IDS:
    print("=" * 80)
    print("Downloading:", repo_id)

    snapshot_path = snapshot_download(
        repo_id=repo_id,
        cache_dir=str(HF_HUB_CACHE),
        max_workers=8,
    )

    downloaded.append({
        "repo_id": repo_id,
        "snapshot_path": snapshot_path,
    })

    print("Cached snapshot:", snapshot_path)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "hf_cache_dir": str(HF_CACHE_DIR),
    "hf_hub_cache": str(HF_HUB_CACHE),
    "models": downloaded,
}

manifest_path = HF_CACHE_DIR / "phi3_download_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("=" * 80)
print("Done. Manifest:", manifest_path)

In [ ]:
# Kiem tra nhanh cac file snapshot da ton tai trong Drive.
from pathlib import Path

for item in downloaded:
    snapshot_dir = Path(item["snapshot_path"])
    safetensors = list(snapshot_dir.glob("*.safetensors"))
    json_files = list(snapshot_dir.glob("*.json"))

    print("=" * 80)
    print(item["repo_id"])
    print("Snapshot:", snapshot_dir)
    print("Exists:", snapshot_dir.exists())
    print("Safetensors files:", len(safetensors))
    print("JSON files:", len(json_files))

## Dung lai model tu Drive

O notebook khac, chi can mount Drive va set lai cac bien cache truoc khi goi `from_pretrained`:

```python
from pathlib import Path
import os

HF_CACHE_DIR = Path('/content/drive/MyDrive/hf_cache')
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR / 'transformers')
```

Sau do `from_pretrained('microsoft/Phi-3-small-8k-instruct')` se uu tien doc tu cache tren Drive neu snapshot da co san.